### Sections
| | |
|---|---|
| **A** | Setup — install, imports, Colab-safe paths |
| **B** | Load artefacts from Notebook 3A |
| **C** | Rebuild ChromaDB collection |
| **D** | Groq API — secure key input + connection test |
| **E** | Build RAG components (retriever · formatter · prompt) |
| **F** | Assemble & test the full RAG chain (LCEL) |
| **G** | Inspect pipeline outputs + handoff to 4B |


### Section A — Setup

In [ ]:
# ── Install all required libraries ───────────────────────────────────────
!pip install -q langchain-core langchain-groq

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
import os, sys, json, warnings, getpass
from pathlib import Path
from dotenv import load_dotenv
from typing  import List, Dict, Any, Optional

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ── Sentence Transformers ─────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import torch

# ── ChromaDB ──────────────────────────────────────────────────────────────────
import chromadb

# ── LangChain Core ────────────────────────────────────────────────────────────
from langchain_core.documents        import Document
from langchain_core.prompts          import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers   import StrOutputParser
from langchain_core.runnables        import RunnablePassthrough, RunnableParallel, RunnableLambda

# ── LangChain Groq ────────────────────────────────────────────────────────────
from langchain_groq import ChatGroq

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"   Device : {DEVICE}")

root = Path.cwd().parent
sys.path.insert(0, str(root))
from config import DATA_DIR


In [ ]:
# ── Path configuration ──────

MODEL_DIR  = DATA_DIR / "models"
OUTPUT_DIR = DATA_DIR / "outputs"

# ── Artifact paths from Notebook 02_01 ───────────────────────────────────────────
FINE_TUNED_MODEL = str(MODEL_DIR / "fine_tuned_embedder")
FT_EMBEDDINGS    = OUTPUT_DIR / "ft_embeddings.npy"
CHUNK_METADATA   = OUTPUT_DIR / "chunk_metadata.json"

# ── Artifacts produced by THIS notebook  ───────────────────
RAG_CONFIG_FILE  = OUTPUT_DIR / "rag_config.json"

COLLECTION_NAME  = "supply_chain_kg"
GROQ_MODEL_NAME  = "openai/gpt-oss-20b"   # updated supported Groq model

print(f" Fine-tuned model: {FINE_TUNED_MODEL}")
print(f" Embeddings file : {FT_EMBEDDINGS}")
print(f" Groq model      : {GROQ_MODEL_NAME}")


### Section B — Load Artifacts from Notebook 02_01

In [ ]:
# ── Load the fine-tuned sentence-transformer embedding model from 02_01 ────────────────
# This model is used ONLY for embedding new user queries at retrieval time.
# The corpus embeddings are loaded from ft_embeddings.npy (pre-computed).
#
# The fine-tuned model is used here for consistency with the indexed embeddings. 
# If you prefer the base model, replace FINE_TUNED_MODEL with "all-MiniLM-L6-v2".
# ─────────────────────────────────────────────────────────────────────────────
ft_model_path = Path(FINE_TUNED_MODEL)
if not ft_model_path.exists():
    raise FileNotFoundError(
        f"Model not found: {FINE_TUNED_MODEL}\n"
        "Please run Notebook 3A first."
    )

embedding_model = SentenceTransformer(FINE_TUNED_MODEL, device=DEVICE)
EMBEDDING_DIM   = embedding_model.get_sentence_embedding_dimension()

print("Embedding model loaded.")
print(f" Source        : {FINE_TUNED_MODEL}")
print(f" Embedding dim : {EMBEDDING_DIM}")


In [ ]:
# ── Load pre-computed embeddings and chunk metadata ───────────────────────
# ft_embeddings   : float32 array (n_chunks × 384), L2-normalised
# metadata_list   : list of dicts with keys: chunk, entity_type, char_length
# ─────────────────────────────────────────────────────────────────────────────

# -- Embeddings --
if not FT_EMBEDDINGS.exists():
    raise FileNotFoundError(
        f"Embeddings not found: {FT_EMBEDDINGS}\nRun Notebook 02_01 first."
    )
ft_embeddings = np.load(FT_EMBEDDINGS)

# -- Metadata --
if not CHUNK_METADATA.exists():
    raise FileNotFoundError(
        f"Metadata not found: {CHUNK_METADATA}\nRun Notebook 02_01 first."
    )
with open(CHUNK_METADATA, "r", encoding="utf-8") as f:
    metadata_list: List[Dict] = json.load(f)

# Unpack for convenience
chunks       = [m["chunk"]       for m in metadata_list]
chunk_labels = [m["entity_type"] for m in metadata_list]

# Sanity check
assert len(chunks) == ft_embeddings.shape[0], (
    f"Length mismatch: {len(chunks)} chunks vs {ft_embeddings.shape[0]} embeddings."
)

print(f"   Chunks          : {len(chunks)}")
print(f"   Embeddings shape: {ft_embeddings.shape}")
print(f"   Entity types    : {sorted(set(chunk_labels))}")


### Section C — Rebuild ChromaDB Collection

ChromaDB is in-memory - it resets between sessions. It is rebuilt from the saved artifacts each time.

In [ ]:
# ── Initialise ChromaDB in-memory client and create collection ─────────────
try:
    chroma_client = chromadb.EphemeralClient()     # chromadb >= 0.4.x
except AttributeError:
    chroma_client = chromadb.Client()              # chromadb <  0.4.x

# Safe re-run: delete if already exists
existing = [c.name for c in chroma_client.list_collections()]
if COLLECTION_NAME in existing:
    chroma_client.delete_collection(COLLECTION_NAME)

collection = chroma_client.create_collection(
    name     = COLLECTION_NAME,
    metadata = {"hnsw:space": "cosine"},
)
print(f"Collection '{COLLECTION_NAME}' created (cosine distance).")


In [ ]:
# ── Index All Embeddings: Add all chunks + embeddings + metadata to ChromaDB ────────────────────
BATCH_SIZE    = 200
doc_ids       = [f"chunk_{i:04d}" for i in range(len(chunks))]
chroma_metas  = [
    {
        "entity_type" : metadata_list[i]["entity_type"],
        "char_length" : metadata_list[i]["char_length"],
        "chunk_index" : i,
    }
    for i in range(len(chunks))
]

print(f"Indexing {len(chunks)} chunks ...")
for start in tqdm(range(0, len(chunks), BATCH_SIZE), desc="Indexing"):
    end = min(start + BATCH_SIZE, len(chunks))
    collection.add(
        ids        = doc_ids[start:end],
        documents  = chunks[start:end],
        embeddings = ft_embeddings[start:end].tolist(),
        metadatas  = chroma_metas[start:end],
    )

assert collection.count() == len(chunks), "Index count mismatch — re-run this cell."
print(f"\nChromaDB ready.  Documents indexed: {collection.count()}")


### Section D — Groq API Setup

In [ ]:
# ── Load Groq API key from .env or prompt if needed ────────────────
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    GROQ_API_KEY = getpass.getpass(
        "Enter your Groq API key (hidden): "
    )
    print("Groq API key received via getpass.")
else:
    print("Groq API key loaded from .env")

if not GROQ_API_KEY.startswith("gsk_"):
    raise ValueError(
        "Key does not start with 'gsk_'. Please check you copied the correct Groq API key."
    )
print(f" Key prefix : {GROQ_API_KEY[:8]}{'*' * 20}  (rest hidden)")


### D2 — Initialise Groq LLM

In [ ]:
# ── Initialise Groq LLM object ───────────────────────────────────────
# temperature=0.1  : low randomness → consistent, factual supply chain answers
# max_tokens=1024  : enough for detailed answers without excessive cost
# ─────────────────────────────────────────────────────────────────────────────
llm = ChatGroq(
    api_key    = GROQ_API_KEY,
    model_name = GROQ_MODEL_NAME,
    temperature= 0.1,
    max_tokens = 1024,
)
print("ChatGroq LLM initialised.")
print(f" Model      : {GROQ_MODEL_NAME}")
print(f" Temperature: 0.1")
print(f" Max tokens : 1024")


In [ ]:
# ── Connection Test: Send a minimal test message to verify the API key works ───────────────
# This catches auth errors early — before we build the full RAG chain.
# ─────────────────────────────────────────────────────────────────────────────
from langchain_core.messages import HumanMessage

print("Testing Groq connection ...")
try:
    test_response = llm.invoke(
        [HumanMessage(content="Reply with exactly: GROQ_OK")]
    )
    reply = test_response.content.strip()
    print(f"Groq API connected successfully.")
    print(f" Model reply : {reply}")
except Exception as e:
    raise ConnectionError(
        f"Groq API connection failed: {e}\n"
        "Check your API key and internet connection."
    )


### Section E — Build RAG Components

Three building blocks assembled here:
1. **Retriever** — queries ChromaDB and returns `Document` objects
2. **Formatter** — converts documents into a context string
3. **Prompt template** — structures the system role + context + question

In [ ]:
# ── Core retrieval function ───────────────────────────────────────────────
# Wraps ChromaDB query into a reusable function that returns
# LangChain Document objects (standard interface for the RAG chain).
#
# Steps:
#   1. Embed the query using the fine-tuned model (normalised)
#   2. Query ChromaDB HNSW index for nearest neighbours
#   3. Convert ChromaDB results → LangChain Document objects
#   4. Similarity = 1 - cosine_distance (ChromaDB returns distance)
# ─────────────────────────────────────────────────────────────────────────────

def retrieve_documents(
    query       : str,
    n_results   : int = 5,
    filter_type : Optional[str] = None,
) -> List[Document]:
    """
    Retrieve the top-n most relevant KG chunks for a query.

    Parameters
    ----------
    query       : Natural language question.
    n_results   : Number of chunks to retrieve (default 5).
    filter_type : Optional entity-type filter e.g. 'Supplier'.

    Returns
    -------
    List of LangChain Document objects, sorted by descending similarity.
    """
    # Step 1 — embed query with the same model used for corpus indexing
    q_embedding: List[float] = embedding_model.encode(
        [query],
        convert_to_numpy     = True,
        normalize_embeddings = True,
    ).tolist()

    # Step 2 — optional metadata filter
    where: Optional[Dict] = {"entity_type": filter_type} if filter_type else None

    # Step 3 — ChromaDB nearest-neighbour search
    results = collection.query(
        query_embeddings = q_embedding,
        n_results        = n_results,
        where            = where,
        include          = ["documents", "distances", "metadatas"],
    )

    # Step 4 — convert to LangChain Documents
    documents: List[Document] = []
    for doc, dist, meta in zip(
        results["documents"][0],
        results["distances"][0],
        results["metadatas"][0],
    ):
        documents.append(Document(
            page_content = doc,
            metadata     = {
                **meta,
                "similarity": round(1.0 - float(dist), 4),
            },
        ))
    return documents


# Quick smoke test
_test_docs = retrieve_documents("Which suppliers have long lead times?", n_results=3)
print("retrieve_documents() defined and tested.")
print(f" Test query returned {len(_test_docs)} documents.")
for d in _test_docs:
    print(f"   [{d.metadata['entity_type']:<16}] sim={d.metadata['similarity']}  "
          f"{d.page_content[:60]}...")


In [ ]:
# ── Document Formatter: Format retrieved documents into a structured context string ────────────
# The context string is inserted into the prompt template.
# Each chunk is numbered and tagged with its entity type and similarity score
# so the LLM can cite sources and understand data provenance.
# ─────────────────────────────────────────────────────────────────────────────

def format_documents(docs: List[Document]) -> str:
    """
    Convert a list of LangChain Documents into a numbered context block.
    Each entry is tagged with entity type and similarity score for
    transparency and source attribution in the LLM response.
    """
    if not docs:
        return "No relevant information found in the supply chain knowledge base."

    lines = ["=== Supply Chain Knowledge Graph Context ==="]
    for i, doc in enumerate(docs, start=1):
        entity_type = doc.metadata.get("entity_type", "Unknown")
        similarity  = doc.metadata.get("similarity",  0.0)
        lines.append(
            f"\n[Context {i}] Entity: {entity_type} | Relevance: {similarity:.3f}"
        )
        lines.append(doc.page_content)
    lines.append("\n=== End of Context ===")
    return "\n".join(lines)


# Smoke test
_fmt = format_documents(_test_docs)
print("format_documents() defined.")
print("\nSample formatted context (first 400 chars):")
print(_fmt[:400])
print("...")


In [ ]:
# ── System Prompt Design ─────────────────────────────────────────────
# The system prompt establishes the LLM's role, constraints, and
# output style for supply chain decision support.
#
# Key design principles:
#   • Role definition      : domain expert persona
#   • Grounding constraint : answer ONLY from provided context
#   • Fallback rule        : acknowledge when context is insufficient
#   • Output format        : structured, actionable answers
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are an expert Supply Chain AI Analyst with deep knowledge of logistics, supplier management, inventory control, and warehouse operations.

You assist supply chain managers by answering questions using information retrieved from the company's Supply Chain Knowledge Graph.

STRICT RULES:
1. Answer ONLY using information from the provided context.
2. If the context does not contain enough information to answer the question, clearly state: "The knowledge base does not contain sufficient information to answer this question fully."
3. Always cite which context entries (e.g. [Context 1], [Context 3]) support your answer.
4. When relevant, include specific values, IDs, scores, and quantities from the context.
5. Structure your answer with: a direct answer first, then supporting details.
6. If recommending an action, briefly explain the reasoning.
7. Keep answers concise and business-focused."""

print(f"Length : {len(SYSTEM_PROMPT)} chars")
print("\nPrompt preview (first 300 chars):")
print(SYSTEM_PROMPT[:300], "...")


In [ ]:
# ── Prompt Template: Build the LangChain ChatPromptTemplate ────────────────────────────────
# Structure:
#   [System]  : role + rules (SYSTEM_PROMPT from E3)
#   [Human]   : context block + user question
#
# Using ChatPromptTemplate.from_messages for chat-format LLMs.
# ─────────────────────────────────────────────────────────────────────────────

RAG_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        SYSTEM_PROMPT,
    ),
    (
        "human",
        "{context}\n\nQuestion: {question}\n\nProvide a clear, structured answer:"
    ),
])

print("RAG prompt template defined.")
print(" Messages :", len(RAG_PROMPT.messages))
print(" Variables:", RAG_PROMPT.input_variables)


### Section F — Assemble & Test the RAG Chain (LCEL)

Using **LangChain Expression Language (LCEL)** — the modern, composable way to build LangChain pipelines.

In [ ]:
# ── Build the RAG chain using LCEL ───────────────────────────────────────
# LCEL pipes (|) connect Runnables into a sequential chain.
# RunnableParallel runs two branches in parallel:
#   - context branch : retrieve docs → format into string
#   - question branch: pass the original query through unchanged
# Both outputs are merged into the prompt template's input dict.
# ─────────────────────────────────────────────────────────────────────────────

# Branch 1: retrieval + formatting (returns str)
retriever_chain = RunnableLambda(
    lambda query: format_documents(retrieve_documents(query, n_results=5))
)

# Branch 2: pass-through for the question (returns str)
question_chain = RunnablePassthrough()

# Parallel merge: {"context": str, "question": str}
retrieval_parallel = RunnableParallel(
    context  = retriever_chain,
    question = question_chain,
)

# Full chain: query → parallel → prompt → LLM → parsed string
rag_chain = retrieval_parallel | RAG_PROMPT | llm | StrOutputParser()

print("RAG chain assembled.")
print("\nChain structure:")
print("  Input (query str)")
print("  │")
print("  ├─ context branch  : retrieve_documents(n=5) → format_documents()")
print("  └─ question branch : RunnablePassthrough()")
print("  │")
print("  ├─ RAG_PROMPT  : SystemMessage + HumanMessage")
print("  ├─ ChatGroq    : openai/gpt-oss-20b (temp=0.1)")
print("  └─ StrOutputParser : extract answer string")


In [ ]:
# ── Run the full RAG chain with one test query ────────────────────────────
# This end-to-end test validates every component is wired correctly:
# embedding model → ChromaDB → formatter → prompt → Groq → parser
# ─────────────────────────────────────────────────────────────────────────────

TEST_QUERY = "Which suppliers have a lead time over 20 days and low reliability?"

print(f"Running RAG chain for test query...")
print(f"Query: \"{TEST_QUERY}\"")
print("─" * 65)

import time
t0 = time.time()
test_answer = rag_chain.invoke(TEST_QUERY)
latency     = round(time.time() - t0, 2)

print(f"\n{'─'*65}")
print("RAG ANSWER:")
print(f"{'─'*65}")
print(test_answer)
print(f"{'─'*65}")
print(f"\nRAG chain test successful.")
print(f"   Latency : {latency}s")
print(f"   Answer  : {len(test_answer)} chars")


In [ ]:
# ── Step-by-step inspection of each pipeline stage ───────────────────────

print("=" * 65)
print("  PIPELINE STAGE INSPECTION")
print("=" * 65)

# Stage 1: Retrieval
print("\n[Stage 1] RETRIEVAL — ChromaDB nearest-neighbour search")
retrieved_docs = retrieve_documents(TEST_QUERY, n_results=5)
for i, doc in enumerate(retrieved_docs, 1):
    print(f"  Doc {i}: [{doc.metadata['entity_type']:<16}] "
          f"sim={doc.metadata['similarity']:.4f}  "
          f"{doc.page_content[:60]}...")

# Stage 2: Formatting
print("\n[Stage 2] FORMATTING — Context block for LLM")
context_str = format_documents(retrieved_docs)
print(context_str[:500])
print("  ...")

# Stage 3: Prompt
print("\n[Stage 3] PROMPT — Final messages sent to LLM")
formatted_prompt = RAG_PROMPT.format_messages(
    context  = context_str,
    question = TEST_QUERY,
)
for msg in formatted_prompt:
    role    = msg.__class__.__name__.replace("Message", "")
    preview = msg.content[:120].replace("\n", " ")
    print(f"  [{role}] {preview}...")

# Stage 4: Answer
print("\n[Stage 4] LLM ANSWER")
print(f"  {test_answer[:200]}...")

print("\nAll pipeline stages inspected successfully.")


### Section G — Save RAG Config & Handoff to 4B

In [ ]:
# ── Save RAG configuration ────────────────────────

rag_config = {
    "groq_model_name"   : GROQ_MODEL_NAME,
    "embedding_model"   : FINE_TUNED_MODEL,
    "collection_name"   : COLLECTION_NAME,
    "n_results_default" : 5,
    "temperature"       : 0.1,
    "max_tokens"        : 1024,
    "system_prompt"     : SYSTEM_PROMPT,
    "total_chunks"      : len(chunks),
    "embedding_dim"     : EMBEDDING_DIM,
}

with open(RAG_CONFIG_FILE, "w", encoding="utf-8") as f:
    json.dump(rag_config, f, indent=2)

print("RAG config saved.")
print(f"   File : {RAG_CONFIG_FILE}")

# ── Final summary ─────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("=" * 65)

print("\n  Components built:")
components = [
    ("Embedding model",    f"{FINE_TUNED_MODEL}"),
    ("ChromaDB collection",f"{COLLECTION_NAME}  ({collection.count()} docs)"),
    ("Retrieval function", "retrieve_documents()"),
    ("Formatter",          "format_documents()"),
    ("System prompt",      f"{len(SYSTEM_PROMPT)} chars"),
    ("Prompt template",    "RAG_PROMPT (system + human)"),
    ("LLM",                f"ChatGroq ({GROQ_MODEL_NAME})"),
    ("RAG chain",          "LCEL: retrieval_parallel | RAG_PROMPT | llm | parser"),
]
for name, detail in components:
    print(f"    {name:<25} : {detail}")

print("\n  Artifacts saved:")
print(f"    {RAG_CONFIG_FILE}  ")
print("=" * 65)
